**Colab version of `CTD_Assignment_6`.** Same code as the Kaggle notebook; the first cell downloads the Kaggle datasets with `kagglehub` (no Kaggle login needed for public datasets). Runtime → Run all to reproduce.

Submission hub with every link for this week: https://github.com/ba-00001/brian-bazurto-python-v2/blob/assignment6/assignment6/README.md

In [ ]:
# Google Colab setup: Kaggle "Add Input" folders don't exist in Colab, so this cell
# downloads the same public Kaggle datasets with kagglehub and copies them into
# /content/kaggle_input (Colab's /kaggle/input is read-only). The rest of the
# notebook reads from KAGGLE_INPUT, so it runs the same as on Kaggle.
import os
import shutil

import numpy as np
import pandas as pd

try:
    import kagglehub
except ImportError:
    %pip install -q kagglehub
    import kagglehub

DATASETS = ['salahuddinahmedshuvo/ecommerce-consumer-behavior-analysis-data', 'samayashar/ai-powered-job-recommendations', 'martj42/international-football-results-from-1872-to-2017']

KAGGLE_INPUT = '/content/kaggle_input'
os.makedirs(KAGGLE_INPUT, exist_ok=True)
for slug in DATASETS:
    downloaded = kagglehub.dataset_download(slug)
    target = os.path.join(KAGGLE_INPUT, slug.split('/')[1])
    shutil.copytree(downloaded, target, dirs_exist_ok=True)

for dirname, _, filenames in os.walk(KAGGLE_INPUT):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Inputs and file lookup

Inputs to add with **Add Input** → Datasets: "Ecommerce Consumer Behavior" by Salahuddin Ahmed (Task 3), "AI-Powered Job Recommendations" by Samay Ashar (Task 4) and "international football results" by Mart Jürisoo (Task 11). The helper below searches `/kaggle/input` for each file, so the exact folder path does not matter; outside Kaggle it looks in a local `data/` folder.

In [ ]:
import glob

INPUT_ROOT = KAGGLE_INPUT
LOCAL_ROOT = 'data'  # local fallback when not running on Kaggle


def find_csvs(columns=None, name=None, sep=','):
    """Return every input CSV whose path contains `name` and whose header has all of `columns`."""
    root = INPUT_ROOT if os.path.isdir(INPUT_ROOT) else LOCAL_ROOT
    found = []
    for path in sorted(glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)):
        if name and name.lower() not in path.lower():
            continue
        if columns:
            try:
                header = pd.read_csv(path, sep=sep, nrows=0).columns.str.strip()
            except Exception:
                continue
            if not set(columns).issubset(header):
                continue
        found.append(path)
    if not found:
        raise FileNotFoundError(f'No input matched name={name!r} columns={columns}. Check that the Kaggle input is added.')
    return found

## Task 1 — Data Selection

Create `df1`, `df2` and `df3` from sample data, then select columns and rows from `df1`.

In [ ]:
data1 = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'Age': [25, 30, 35, 40, 30],
    'Salary': [50000, 60000, 70000, 80000, 55000]
}
data2 = {
    'Name': ['Frank', 'Grace', 'Helen', 'Ian', 'Jack'],
    'Age': [28, 33, 35, 29, 40],
    'Salary': [52000, 58000, 72000, 61000, 85000]
}
data3 = {
    'Name': ['Frank', 'Helen', 'Ian', 'Hima', 'Chaka'],
    'Age': [17, 93, 12, 57, 106],
    'Favorite Color': ['blue', 'pink', 'burgundy', 'red', 'turquoise']
}
df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)
df3 = pd.DataFrame(data3)
print(df1, df2, df3, sep='\n\n')

In [ ]:
print(df1['Name'])
print()
print(df1[['Name', 'Salary']])
print()
print(df1.iloc[0:3])

## Task 2 — Data Aggregation

Group `df1` by `Age` and get the mean, sum and count of `Salary` for each age.

In [ ]:
salary_by_age = df1.groupby('Age')['Salary'].agg(['mean', 'sum', 'count'])
print(salary_by_age)

## Task 3 — Practice Pivot Tables

Read the ecommerce data, turn `Purchase_Amount` from a dollar string into a float, and build the `buying_patterns` and `demographics` pivot tables.

In [ ]:
ecommerce_path = find_csvs(columns=['Customer_ID', 'Purchase_Category', 'Purchase_Amount'])[0]
print(ecommerce_path)
ecommerce = pd.read_csv(ecommerce_path)
print(ecommerce.head())

In [ ]:
# "$333.80" -> 333.8 ; regex=False so $ is just a character
ecommerce['Purchase_Amount'] = (
    ecommerce['Purchase_Amount'].str.replace('$', '', regex=False).str.strip().astype(float)
)
print(ecommerce['Purchase_Amount'].head())

In [ ]:
buying_patterns = ecommerce.pivot_table(
    index='Purchase_Category',
    columns=['Gender', 'Income_Level'],
    values='Purchase_Amount',
    aggfunc='sum',
)
print(buying_patterns)

In [ ]:
demographics = ecommerce.pivot_table(
    index=['Income_Level', 'Education_Level'],
    columns='Marital_Status',
    values='Customer_ID',
    aggfunc='count',
)
print(demographics)

## Task 4 — Practice apply()

Read the job recommendations data into `jobs`, flag entry level jobs paying at least 70000 that need both SQL and Python, and keep those in `my_jobs`.

In [ ]:
jobs_path = find_csvs(name='job')[0]
print(jobs_path)
jobs = pd.read_csv(jobs_path)
print(jobs.head(10))

In [ ]:
# look up the column names once so the check below stays readable
level_col = [c for c in jobs.columns if 'experience' in c.lower()][0]
salary_col = [c for c in jobs.columns if 'salary' in c.lower()][0]
skills_col = [c for c in jobs.columns if 'skill' in c.lower()][0]
print(level_col, '|', salary_col, '|', skills_col)

# make sure salary is a number (strip $ and commas if it was read as text)
if jobs[salary_col].dtype == object:
    jobs[salary_col] = pd.to_numeric(
        jobs[salary_col].str.replace(r'[$,]', '', regex=True), errors='coerce'
    )


def check_job(row):
    skills = [s.strip().lower() for s in str(row[skills_col]).split(',')]
    entry_level = 'entry' in str(row[level_col]).lower()
    if entry_level and row[salary_col] >= 70000 and 'sql' in skills and 'python' in skills:
        return 'Yes'
    return 'No'


jobs['Check These Out'] = jobs.apply(check_job, axis=1)
print(jobs['Check These Out'].value_counts())

In [ ]:
my_jobs = jobs[jobs['Check These Out'] == 'Yes']
print(my_jobs.head(10))

## Task 5 — Merging and Joining DataFrames

Outer merge `df1` and `df3` on `Name`, fill the gaps, build one `Age` column with `np.where`, then do the same kind of combine with `join()`.

In [ ]:
df_1_3_merged = pd.merge(df1, df3, on='Name', how='outer', suffixes=['_left', '_right'])
print(df_1_3_merged)

In [ ]:
df_1_3_merged['Salary'] = df_1_3_merged['Salary'].fillna(15000)
df_1_3_merged['Favorite Color'] = df_1_3_merged['Favorite Color'].fillna('yellow')
print(df_1_3_merged)

In [ ]:
df_1_3_merged['Age'] = np.where(df_1_3_merged['Age_left'].notna(),df_1_3_merged['Age_left'], df_1_3_merged['Age_right'])
# This works like a ternary expression.  If the expression passed to np.where is True, you get the left value, otherwise the right value.
print(df_1_3_merged)

In [ ]:
df_1_3_merged = df_1_3_merged.drop(columns=['Age_left', 'Age_right'])
print(df_1_3_merged)

In [ ]:
df1_b = df1.set_index('Name')
df3_b = df3.set_index('Name')

# join has no default suffixes, so Age needs lsuffix/rsuffix
joined = df1_b.join(df3_b, how='outer', lsuffix='_left', rsuffix='_right')
print(joined)

## Task 6 — Filtering Rows Based on Conditions

Rows of `df1` where `Age` is over 30.

In [ ]:
print(df1[df1['Age'] > 30])

## Task 7 — Sorting Data

`df1` sorted by `Salary`, highest first.

In [ ]:
print(df1.sort_values('Salary', ascending=False))

## Task 8 — Renaming Columns

Rename `Age` and `Salary` into a new DataFrame so `df1` itself is unchanged.

In [ ]:
df1_renamed = df1.rename(columns={'Age': 'Employee Age', 'Salary': 'Employee Salary'})
print(df1_renamed)

## Task 9 — Data Transformation

Give every employee in `df1` a 10% raise.

In [ ]:
df1['Salary'] = df1['Salary'] * 1.1
print(df1)

## Task 10 — Concatenating DataFrames

Add the rows of `df2` under `df1` with a fresh index.

In [ ]:
df_1_2 = pd.concat([df1, df2], ignore_index=True)
print(df_1_2)

## Task 11 — Data Wrangling a Kaggle Dataset

Reshape the international football results so each game shows up once per team, then find the teams that give up the most goals on average.

In [ ]:
results_path = [p for p in find_csvs(name='results') if os.path.basename(p) == 'results.csv'][0]
print(results_path)
football_results = pd.read_csv(results_path)
print(football_results.head())

In [ ]:
results_1 = football_results[['home_team', 'away_team', 'home_score', 'away_score', 'date']]
print(results_1.head())

In [ ]:
# home team point of view
results_2 = results_1.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_score': 'points_for',
    'away_score': 'points_against',
})
print(results_2.head())

In [ ]:
# away team point of view
results_3 = results_1.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_score': 'points_for',
    'home_score': 'points_against',
})
print(results_3.head())

In [ ]:
football_results = pd.concat([results_2, results_3], ignore_index=True)
print(football_results.head())

In [ ]:
points_against = football_results.groupby('team')['points_against'].mean()
points_against = points_against.sort_values(ascending=False)
print(points_against.head(10))

## Task 12 — More Data Wrangling for Football Results

The 10 most recent games for Tunisia, newest first.

In [ ]:
# dates are YYYY-MM-DD strings, but converting makes the sort safe
tunisia = football_results[football_results['team'] == 'Tunisia'].copy()
tunisia['date'] = pd.to_datetime(tunisia['date'])
tunisia = tunisia.dropna(subset=['points_for'])  # skip any scheduled games with no score yet
tunisia_recent = tunisia.sort_values('date', ascending=False).head(10)
print(tunisia_recent)